In [164]:
# Biblioteker
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects
from scipy.stats import chi2
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

In [165]:
# Indlæs data
GDP = pd.read_csv("Data/GDP.csv", skiprows=4)
ODR = pd.read_csv("Data/age-dependency-ratio-old.csv")
TFP = pd.read_csv("Data/total-factor-productivity.csv")
HC_DATA = pd.read_excel("Data/pwt110.xlsx", sheet_name="Data")
HC = HC_DATA[['country', 'year', 'hc', 'ctfp', 'rgdpe', 'pop']]
GEO = pd.read_excel("Data/geo_cepii.xls")
OPN = pd.read_csv("Data/Oppenness.csv", skiprows=4)
URB = pd.read_csv("Data/Urbanization.csv", skiprows=4)

# Sammensæt data
Formålet er at konstruere et tværsnitsdatasæt med ét datapunkt pr. land, hvor vi forklarer TFP-vækst fra 2002 til 2020 med initial ODR og kontroller fra 2002

In [166]:
# Relevante variabler fra HC_DATA
HC = HC_DATA[['country','year','hc','ctfp','rgdpe','pop']].copy()

# Relevante geografiske variable
GEO = GEO[['country', 'landlocked', 'continent']].copy()

# Trade openness i 2002
opn2002 = OPN[['Country Name', '2002']].copy()
opn2002.columns = ['country', 'TradeOpen2002']

# Urbanisering i 2002
URB = URB[['Country Name', '2002']].copy()
URB.columns = ['country', 'Urban2002']
URB['Urban2002'] = pd.to_numeric(URB['Urban2002'], errors='coerce')

# BNP per capita
HC['gdp_pc'] = HC['rgdpe'] / HC['pop']

# Initiale variable i 2002
base2002 = HC[HC['year'] == 2002][['country','hc','ctfp','gdp_pc']].copy()
base2002.columns = ['country','HC2002','TFP2002','GDPpc2002']

# TFP i 2020
tfp2020 = HC[HC['year'] == 2020][['country','ctfp']].copy()
tfp2020.columns = ['country','TFP2020']

# Long-difference datasæt
Tvaersnit = pd.merge(base2002, tfp2020, on='country', how='inner')
Tvaersnit['GrowthTFP'] = np.log(Tvaersnit['TFP2020']) - np.log(Tvaersnit['TFP2002'])

# ODR i 2002
odr2002 = ODR[ODR['Year'] == 2002][[
    'Entity',
    'Age dependency ratio, old (% of working-age population)'
]].copy()
odr2002.columns = ['country', 'ODR2002']

# Samlet tværsnit
Tvaersnit = pd.merge(Tvaersnit, odr2002, on='country', how='inner')
Tvaersnit = pd.merge(Tvaersnit, GEO, on='country', how='left')
Tvaersnit = pd.merge(Tvaersnit, opn2002, on='country', how='left')
Tvaersnit = pd.merge(Tvaersnit, URB, on='country', how='left')

# Fjern manglende observationer
Tvaersnit = Tvaersnit.dropna()

# Deskriptiv statistik

In [167]:
print(Tvaersnit.describe())

          HC2002    TFP2002     GDPpc2002    TFP2020  GrowthTFP    ODR2002  \
count  97.000000  97.000000     97.000000  97.000000  97.000000  97.000000   
mean    2.495553   0.685673  19505.174638   0.644302  -0.025700  12.830334   
std     0.670612   0.287543  18159.820584   0.210707   0.266992   7.836286   
min     1.088122   0.167584   1048.418252   0.200132  -0.515328   1.583193   
25%     2.098030   0.446838   4909.995922   0.475661  -0.222283   6.080334   
50%     2.538953   0.662850  11713.944910   0.622159  -0.057211   8.787122   
75%     2.971450   0.884870  34817.300051   0.821492   0.157685  20.612047   
max     3.583555   1.394810  84592.225755   1.078187   0.942964  28.059470   

       landlocked  TradeOpen2002   Urban2002  
count   97.000000      97.000000   97.000000  
mean     0.154639      77.711973   61.742295  
std      0.363439      46.988081   21.294589  
min      0.000000      20.447122   14.610784  
25%      0.000000      51.609565   43.782407  
50%      0.0000

# ODR + geo variabler

Growth_TFP = α + β1 ODR2002 + β2 TradeOpen2002 + β3 Urban2002 + β4 landlocked + ε

In [168]:
# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'ODR2002',
    'Urban2002',
    'landlocked',
    'TradeOpen2002'
]].dropna().copy()

# Antal lande og observationer
antal_lande = df_model['country'].nunique()
antal_observationer = df_model.shape[0]

# Automatisk print
print("=" * 80)
print("MODEL:")
print("GrowthTFP = α + β1 ODR2002 + β2 TradeOpen2002 + β3 Urban2002 + β4 landlocked + ε")
print(f"Antal lande: {antal_lande}")
print(f"Antal observationer: {antal_observationer}")
print("=" * 80)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): Kun ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): + TradeOpen2002
X2 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + log_area
X3 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + landlocked
X4 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Tabel
results = summary_col(
    [model1, model2, model3, model4],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)'],
    regressor_order=['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)

MODEL:
GrowthTFP = α + β1 ODR2002 + β2 TradeOpen2002 + β3 Urban2002 + β4 landlocked + ε
Antal lande: 91
Antal observationer: 97

                  (1)        (2)        (3)       (4)   
--------------------------------------------------------
ODR2002        -0.0084*** -0.0084*** -0.0094** -0.0094**
               (0.0030)   (0.0030)   (0.0037)  (0.0037) 
TradeOpen2002             -0.0002    -0.0003   -0.0003  
                          (0.0004)   (0.0004)  (0.0004) 
Urban2002                            0.0008    0.0007   
                                     (0.0014)  (0.0016) 
landlocked                                     -0.0226  
                                               (0.0919) 
R-squared      0.0607     0.0619     0.0651    0.0659   
R-squared Adj. 0.0508     0.0420     0.0349    0.0253   
N              97         97         97        97       
R2             0.061      0.062      0.065     0.066    
Standard errors in parentheses.
* p<.1, ** p<.05, ***p<.01


# HC + geo variabler

Growth_TFP = α + β1 ODR2002 + β2 HC2002 + β3 TradeOpen2002 + β4 Urban2002 + β5 landlocked + ε

In [169]:
# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'ODR2002',
    'HC2002',
    'TradeOpen2002',
    'Urban2002',
    'landlocked'
]].dropna().copy()

# Antal lande og observationer
antal_lande = df_model['country'].nunique()
antal_observationer = df_model.shape[0]

# Automatisk print
print("=" * 95)
print("MODEL:")
print("GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 TradeOpen2002 + β4 Urban2002 + β5 landlocked + ε")
print(f"Antal lande: {antal_lande}")
print(f"Antal observationer: {antal_observationer}")
print("=" * 95)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): const + ODR + HC2002
X2 = sm.add_constant(df_model[['ODR2002', 'HC2002']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + TradeOpen2002
X3 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'TradeOpen2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + Urban2002
X4 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'TradeOpen2002', 'Urban2002']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + landlocked
X5 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'TradeOpen2002', 'Urban2002', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

# Samlet tabel
results = summary_col(
    [model1, model2, model3, model4, model5],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)'],
    regressor_order=['ODR2002', 'HC2002', 'TradeOpen2002', 'Urban2002', 'landlocked'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)

MODEL:
GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 TradeOpen2002 + β4 Urban2002 + β5 landlocked + ε
Antal lande: 91
Antal observationer: 97

                  (1)        (2)        (3)        (4)        (5)    
---------------------------------------------------------------------
ODR2002        -0.0084*** -0.0121*** -0.0131*** -0.0132*** -0.0133***
               (0.0030)   (0.0046)   (0.0048)   (0.0048)   (0.0047)  
HC2002                    0.0574     0.0728     0.0686     0.0718    
                          (0.0661)   (0.0713)   (0.0797)   (0.0776)  
TradeOpen2002                        -0.0004    -0.0004    -0.0004   
                                     (0.0004)   (0.0004)   (0.0004)  
Urban2002                                       0.0002     0.0000    
                                                (0.0016)   (0.0017)  
landlocked                                                 -0.0316   
                                                           (0.0880)  
R-squared      0.06

# Log(GDP) + geo variabler 

Growth_TFP = α + β1 ODR2002 + β2 log(GDPpc2002) + β3 TradeOpen2002 + β4 Urban2002 + β5 landlocked + ε


In [170]:
# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'ODR2002',
    'GDPpc2002',
    'TradeOpen2002',
    'Urban2002',
    'landlocked'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])

# Antal lande og observationer
antal_lande = df_model['country'].nunique()
antal_observationer = df_model.shape[0]

# Automatisk print
print("=" * 100)
print("MODEL:")
print("GrowthTFP = α + β1 ODR2002 + β2 log(GDPpc2002) + β3 TradeOpen2002 + β4 Urban2002 + β5 landlocked + ε")
print(f"Antal lande: {antal_lande}")
print(f"Antal observationer: {antal_observationer}")
print("=" * 100)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): const + ODR + log_GDPpc2002
X2 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + TradeOpen2002
X3 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002', 'TradeOpen2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + Urban2002
X4 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002', 'TradeOpen2002', 'Urban2002']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + landlocked
X5 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002', 'TradeOpen2002', 'Urban2002', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

# Samlet tabel
results = summary_col(
    [model1, model2, model3, model4, model5],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)'],
    regressor_order=['ODR2002', 'log_GDPpc2002', 'TradeOpen2002', 'Urban2002', 'landlocked'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)

MODEL:
GrowthTFP = α + β1 ODR2002 + β2 log(GDPpc2002) + β3 TradeOpen2002 + β4 Urban2002 + β5 landlocked + ε
Antal lande: 91
Antal observationer: 97

                  (1)        (2)       (3)       (4)        (5)    
-------------------------------------------------------------------
ODR2002        -0.0084*** -0.0020   -0.0013   0.0016     0.0019    
               (0.0030)   (0.0041)  (0.0042)  (0.0041)   (0.0041)  
log_GDPpc2002             -0.0632** -0.0707** -0.1793*** -0.1828***
                          (0.0319)  (0.0361)  (0.0554)   (0.0550)  
TradeOpen2002                       0.0003    0.0005     0.0006    
                                    (0.0004)  (0.0004)   (0.0005)  
Urban2002                                     0.0064***  0.0062*** 
                                              (0.0021)   (0.0022)  
landlocked                                               -0.0524   
                                                         (0.0849)  
R-squared      0.0607     0.1017   

# Regioner + Geo variabler

GrowthTFP = α + β1 ODR2002 + β2 Europe + β3 Asia + β4 Africa + β5 LatinAmerica + β6 NorthAmerica + β7 Oceania + β8 MiddleEast + β9 TradeOpen2002 + β10 Urban2002 + β11 landlocked + ε


In [171]:
# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'ODR2002',
    'GDPpc2002',
    'TradeOpen2002',
    'Urban2002',
    'landlocked'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])

# Antal lande og observationer
antal_lande = df_model['country'].nunique()
antal_observationer = df_model.shape[0]

# Automatisk print
print("=" * 185)
print("MODEL:")
print("GrowthTFP = α + β1 ODR2002 + β2 Europe + β3 Asia + β4 Africa + β5 LatinAmerica + β6 NorthAmerica + β7 Oceania + β8 MiddleEast + β9 TradeOpen2002 + β10 Urban2002 + β11 landlocked + ε")
print(f"Antal lande: {antal_lande}")
print(f"Antal observationer: {antal_observationer}")
print("=" * 185)

# Regioner
europe = [
    'Albania','Austria','Belgium','Bulgaria','Croatia','Cyprus','Czechia','Denmark','Estonia','Finland','France','Germany',
    'Greece','Hungary','Iceland','Ireland','Italy','Latvia','Lithuania','Luxembourg','Malta','Netherlands','Norway',
    'Poland','Portugal','Romania','Serbia','Slovakia','Slovenia','Spain','Sweden','Switzerland','Ukraine','United Kingdom','Armenia']

asia = [
    'Bahrain','China','India','Indonesia','Iraq','Israel','Japan','Jordan','Kazakhstan','Kuwait','Kyrgyzstan','Malaysia',
    'Mongolia','Philippines','Qatar','Saudi Arabia','Singapore','Sri Lanka','Taiwan','Tajikistan','Thailand']

africa = [
    'Angola','Benin','Botswana','Burkina Faso','Burundi','Cameroon','Central African Republic','Egypt','Eswatini',
    'Gabon','Kenya','Lesotho','Mauritania','Mauritius','Morocco','Mozambique','Namibia','Niger','Nigeria',
    'Rwanda','Senegal','Sierra Leone','South Africa','Sudan','Togo','Tunisia','Zambia','Zimbabwe']

latin_america = [
    'Argentina','Barbados','Brazil','Chile','Colombia','Costa Rica','Dominican Republic','Ecuador','El Salvador',
    'Guatemala','Honduras','Jamaica','Mexico','Nicaragua','Panama','Paraguay','Peru','Trinidad and Tobago','Uruguay']

north_america = ['Canada', 'United States']

oceania = ['Australia', 'Fiji', 'New Zealand']

middle_east = ['Bahrain', 'Iraq', 'Israel', 'Jordan', 'Kuwait', 'Qatar', 'Saudi Arabia']

# Regionale dummies
df_model['Europe'] = df_model['country'].isin(europe).astype(int)
df_model['Asia'] = df_model['country'].isin(asia).astype(int)
df_model['Africa'] = df_model['country'].isin(africa).astype(int)
df_model['LatinAmerica'] = df_model['country'].isin(latin_america).astype(int)
df_model['NorthAmerica'] = df_model['country'].isin(north_america).astype(int)
df_model['Oceania'] = df_model['country'].isin(oceania).astype(int)
df_model['MiddleEast'] = df_model['country'].isin(middle_east).astype(int)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): const + ODR + regionale dummies
X2 = sm.add_constant(df_model[['ODR2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + TradeOpen2002
X3 = sm.add_constant(df_model[['ODR2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast', 'TradeOpen2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + Urban2002
X4 = sm.add_constant(df_model[['ODR2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast',
                               'TradeOpen2002', 'Urban2002']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + landlocked
X5 = sm.add_constant(df_model[['ODR2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast',
                               'TradeOpen2002', 'Urban2002', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

results = summary_col(
    [model1, model2, model3, model4, model5],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)'],
    regressor_order=[
        'ODR2002', 'Europe', 'Asia', 'Africa',
        'LatinAmerica', 'NorthAmerica', 'Oceania',
        'MiddleEast', 'TradeOpen2002', 'Urban2002',
        'landlocked'
    ],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)

MODEL:
GrowthTFP = α + β1 ODR2002 + β2 Europe + β3 Asia + β4 Africa + β5 LatinAmerica + β6 NorthAmerica + β7 Oceania + β8 MiddleEast + β9 TradeOpen2002 + β10 Urban2002 + β11 landlocked + ε
Antal lande: 91
Antal observationer: 97

                  (1)       (2)      (3)       (4)       (5)   
---------------------------------------------------------------
ODR2002        -0.0084*** -0.0078  -0.0095  -0.0158*  -0.0159* 
               (0.0030)   (0.0074) (0.0079) (0.0090)  (0.0090) 
Europe                    -0.0130  0.0168   0.0450    0.0476   
                          (0.0925) (0.1036) (0.1034)  (0.1010) 
Asia                      0.1137** 0.1346** 0.1655*** 0.1650***
                          (0.0521) (0.0554) (0.0559)  (0.0571) 
Africa                    -0.0149  -0.0272  -0.0179   -0.0167  
                          (0.0794) (0.0849) (0.0852)  (0.0877) 
LatinAmerica              -0.0076  -0.0136  -0.0714   -0.0713  
                          (0.0618) (0.0632) (0.0763)  (0.0773) 
No

# HC + log(GDP) + regionale dummies + geo variabler

In [172]:
# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'HC2002',
    'ODR2002',
    'GDPpc2002',
    'TradeOpen2002',
    'Urban2002',
    'landlocked'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])

# Antal lande og observationer
antal_lande = df_model['country'].nunique()
antal_observationer = df_model.shape[0]

# Automatisk print
print("=" * 220)
print("MODEL:")
print("GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 log(GDPpc2002) + β4 Europe + β5 Asia + β6 Africa + β7 LatinAmerica + β8 NorthAmerica + β9 Oceania + β10 MiddleEast + β11 TradeOpen2002 + β12 Urban2002 + β13 landlocked + ε")
print(f"Antal lande: {antal_lande}")
print(f"Antal observationer: {antal_observationer}")
print("=" * 220)

# Regioner
europe = [
    'Albania','Austria','Belgium','Bulgaria','Croatia','Cyprus','Czechia','Denmark','Estonia','Finland','France','Germany',
    'Greece','Hungary','Iceland','Ireland','Italy','Latvia','Lithuania','Luxembourg','Malta','Netherlands','Norway',
    'Poland','Portugal','Romania','Serbia','Slovakia','Slovenia','Spain','Sweden','Switzerland','Ukraine','United Kingdom','Armenia']

asia = [
    'Bahrain','China','India','Indonesia','Iraq','Israel','Japan','Jordan','Kazakhstan','Kuwait','Kyrgyzstan','Malaysia',
    'Mongolia','Philippines','Qatar','Saudi Arabia','Singapore','Sri Lanka','Taiwan','Tajikistan','Thailand']

africa = [
    'Angola','Benin','Botswana','Burkina Faso','Burundi','Cameroon','Central African Republic','Egypt','Eswatini',
    'Gabon','Kenya','Lesotho','Mauritania','Mauritius','Morocco','Mozambique','Namibia','Niger','Nigeria',
    'Rwanda','Senegal','Sierra Leone','South Africa','Sudan','Togo','Tunisia','Zambia','Zimbabwe']

latin_america = [
    'Argentina','Barbados','Brazil','Chile','Colombia','Costa Rica','Dominican Republic','Ecuador','El Salvador',
    'Guatemala','Honduras','Jamaica','Mexico','Nicaragua','Panama','Paraguay','Peru','Trinidad and Tobago','Uruguay']

north_america = ['Canada', 'United States']

oceania = ['Australia', 'Fiji', 'New Zealand']

middle_east = ['Bahrain', 'Iraq', 'Israel', 'Jordan', 'Kuwait', 'Qatar', 'Saudi Arabia']

# Regionale dummies
df_model['Europe'] = df_model['country'].isin(europe).astype(int)
df_model['Asia'] = df_model['country'].isin(asia).astype(int)
df_model['Africa'] = df_model['country'].isin(africa).astype(int)
df_model['LatinAmerica'] = df_model['country'].isin(latin_america).astype(int)
df_model['NorthAmerica'] = df_model['country'].isin(north_america).astype(int)
df_model['Oceania'] = df_model['country'].isin(oceania).astype(int)
df_model['MiddleEast'] = df_model['country'].isin(middle_east).astype(int)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): + kontroller
X2 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + kontroller
X3 = sm.add_constant(df_model[['ODR2002', 'Urban2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + kontroller
X4 = sm.add_constant(df_model[['ODR2002','landlocked']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + kontroller
X5 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

# Model (6): + kontroller
X6 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','HC2002']])
model6 = sm.OLS(y, X6).fit(cov_type='HC1')

# Model (7): + kontroller
X7 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','log_GDPpc2002']])
model7 = sm.OLS(y, X7).fit(cov_type='HC1')

# Model (8): + TradeOpen2002
X8 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','Europe', 'Asia', 'Africa','LatinAmerica', 'NorthAmerica','Oceania', 'MiddleEast']])
model8 = sm.OLS(y, X8).fit(cov_type='HC1')

# Model (9): + TradeOpen2002
X9 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','HC2002','log_GDPpc2002','Europe', 'Asia', 'Africa','LatinAmerica', 'NorthAmerica','Oceania', 'MiddleEast']])
model9 = sm.OLS(y, X9).fit(cov_type='HC1')

results = summary_col(
    [model1, model2, model3, model4, model5, model6, model7, model8, model9],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)', '(6)', '(7)', '(8)', '(9)'],
    regressor_order=[
        'ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','HC2002', 'log_GDPpc2002',
        'Europe', 'Asia', 'Africa', 'LatinAmerica',
        'NorthAmerica', 'Oceania', 'MiddleEast'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)

MODEL:
GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 log(GDPpc2002) + β4 Europe + β5 Asia + β6 Africa + β7 LatinAmerica + β8 NorthAmerica + β9 Oceania + β10 MiddleEast + β11 TradeOpen2002 + β12 Urban2002 + β13 landlocked + ε
Antal lande: 91
Antal observationer: 97

                  (1)        (2)        (3)       (4)        (5)       (6)        (7)        (8)       (9)    
--------------------------------------------------------------------------------------------------------------
ODR2002        -0.0084*** -0.0084*** -0.0092** -0.0086*** -0.0094** -0.0133*** 0.0019     -0.0159*  -0.0090   
               (0.0030)   (0.0030)   (0.0036)  (0.0030)   (0.0037)  (0.0047)   (0.0041)   (0.0090)  (0.0071)  
TradeOpen2002             -0.0002                         -0.0003   -0.0004    0.0006     -0.0009   -0.0002   
                          (0.0004)                        (0.0004)  (0.0004)   (0.0005)   (0.0006)  (0.0006)  
Urban2002                            0.0006               0.0007    0.

# HC + log(GDP) + regionale dummies + geo variabler

GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 log(GDPpc2002) + β4 Europe + β5 Asia + β6 Africa + β7 LatinAmerica + β8 NorthAmerica + β9 Oceania + β10 MiddleEast + β11 TradeOpen2002 + β12 Urban2002 + β13 landlocked + ε

In [173]:
# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'HC2002',
    'ODR2002',
    'GDPpc2002',
    'TradeOpen2002',
    'Urban2002',
    'landlocked'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])

# Antal lande og observationer
antal_lande = df_model['country'].nunique()
antal_observationer = df_model.shape[0]

# Automatisk print
print("=" * 220)
print("MODEL:")
print("GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 log(GDPpc2002) + β4 Europe + β5 Asia + β6 Africa + β7 LatinAmerica + β8 NorthAmerica + β9 Oceania + β10 MiddleEast + β11 TradeOpen2002 + β12 Urban2002 + β13 landlocked + ε")
print(f"Antal lande: {antal_lande}")
print(f"Antal observationer: {antal_observationer}")
print("=" * 220)

# Regioner
europe = [
    'Albania','Austria','Belgium','Bulgaria','Croatia','Cyprus','Czechia','Denmark','Estonia','Finland','France','Germany',
    'Greece','Hungary','Iceland','Ireland','Italy','Latvia','Lithuania','Luxembourg','Malta','Netherlands','Norway',
    'Poland','Portugal','Romania','Serbia','Slovakia','Slovenia','Spain','Sweden','Switzerland','Ukraine','United Kingdom','Armenia']

asia = [
    'Bahrain','China','India','Indonesia','Iraq','Israel','Japan','Jordan','Kazakhstan','Kuwait','Kyrgyzstan','Malaysia',
    'Mongolia','Philippines','Qatar','Saudi Arabia','Singapore','Sri Lanka','Taiwan','Tajikistan','Thailand']

africa = [
    'Angola','Benin','Botswana','Burkina Faso','Burundi','Cameroon','Central African Republic','Egypt','Eswatini',
    'Gabon','Kenya','Lesotho','Mauritania','Mauritius','Morocco','Mozambique','Namibia','Niger','Nigeria',
    'Rwanda','Senegal','Sierra Leone','South Africa','Sudan','Togo','Tunisia','Zambia','Zimbabwe']

latin_america = [
    'Argentina','Barbados','Brazil','Chile','Colombia','Costa Rica','Dominican Republic','Ecuador','El Salvador',
    'Guatemala','Honduras','Jamaica','Mexico','Nicaragua','Panama','Paraguay','Peru','Trinidad and Tobago','Uruguay']

north_america = ['Canada', 'United States']

oceania = ['Australia', 'Fiji', 'New Zealand']

middle_east = ['Bahrain', 'Iraq', 'Israel', 'Jordan', 'Kuwait', 'Qatar', 'Saudi Arabia']

# Regionale dummies
df_model['Europe'] = df_model['country'].isin(europe).astype(int)
df_model['Asia'] = df_model['country'].isin(asia).astype(int)
df_model['Africa'] = df_model['country'].isin(africa).astype(int)
df_model['LatinAmerica'] = df_model['country'].isin(latin_america).astype(int)
df_model['NorthAmerica'] = df_model['country'].isin(north_america).astype(int)
df_model['Oceania'] = df_model['country'].isin(oceania).astype(int)
df_model['MiddleEast'] = df_model['country'].isin(middle_east).astype(int)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): + kontroller
X2 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + kontroller
X3 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked', 'HC2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + kontroller
X4 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked', 'HC2002', 'log_GDPpc2002']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + kontroller
X5 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','HC2002', 'log_GDPpc2002', 'Europe', 'Asia', 'Africa','LatinAmerica', 'NorthAmerica','Oceania', 'MiddleEast']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')


results = summary_col(
    [model1, model2, model3, model4, model5],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)'],
    regressor_order=[
        'ODR2002', 'TradeOpen2002', 'Urban2002', 'landlocked','HC2002', 'log_GDPpc2002',
        'Europe', 'Asia', 'Africa', 'LatinAmerica',
        'NorthAmerica', 'Oceania', 'MiddleEast'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)

MODEL:
GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 log(GDPpc2002) + β4 Europe + β5 Asia + β6 Africa + β7 LatinAmerica + β8 NorthAmerica + β9 Oceania + β10 MiddleEast + β11 TradeOpen2002 + β12 Urban2002 + β13 landlocked + ε
Antal lande: 91
Antal observationer: 97

                  (1)        (2)       (3)        (4)        (5)    
--------------------------------------------------------------------
ODR2002        -0.0084*** -0.0094** -0.0133*** -0.0050    -0.0090   
               (0.0030)   (0.0037)  (0.0047)   (0.0049)   (0.0071)  
TradeOpen2002             -0.0003   -0.0004    0.0004     -0.0002   
                          (0.0004)  (0.0004)   (0.0005)   (0.0006)  
Urban2002                 0.0007    0.0000     0.0058***  0.0082*** 
                          (0.0016)  (0.0017)   (0.0021)   (0.0030)  
landlocked                -0.0226   -0.0316    -0.0796    -0.0763   
                          (0.0919)  (0.0880)   (0.0789)   (0.0832)  
HC2002                              0.0718    